# Lab 02: Webhook Integration - Fundamentals

**Lab**: 02-webhook-integration  
**Duration**: ~5 minutes  
**Prerequisites**: Stripe account, basic Python knowledge

## Learning Objectives

By the end of this notebook, you will:
- Understand what webhooks are and why they matter
- Know common Stripe event types
- Understand webhook signature verification

---

## What are Webhooks?

**Webhooks** are HTTP callbacks that notify your application when events happen in Stripe.

### Push vs Pull Model

| Polling (Pull) | Webhooks (Push) |
|---------------|------------------|
| Your app asks "any updates?" repeatedly | Stripe tells you when something happens |
| Wastes resources checking constantly | Efficient - only notified when needed |
| May miss events between polls | Real-time notifications |
| Adds latency | Instant response to events |

## How Webhooks Work

```
1. Event Occurs in Stripe
   +------------------+
   | Customer pays    |
   | for subscription |
   +------------------+
           |
           v
2. Stripe Sends HTTP POST
   +------------------+      HTTP POST        +------------------+
   |                  | -------------------> |                  |
   |   Stripe API     |   Event payload     |   Your Server    |
   |                  | <------------------ |                  |
   +------------------+      200 OK         +------------------+
           
3. Your App Processes Event
   +------------------+
   | - Verify sig     |
   | - Update DB      |
   | - Send email     |
   +------------------+
```

## Common Stripe Event Types

Stripe has [hundreds of event types](https://stripe.com/docs/api/events/types). Here are the most common:

### Payment Events

| Event | When it fires |
|-------|---------------|
| `payment_intent.succeeded` | Payment completed successfully |
| `payment_intent.payment_failed` | Payment attempt failed |
| `charge.refunded` | A charge was refunded |
| `charge.dispute.created` | Customer disputed a charge |

### Subscription Events

| Event | When it fires |
|-------|---------------|
| `customer.subscription.created` | New subscription started |
| `customer.subscription.updated` | Subscription changed (plan, quantity) |
| `customer.subscription.deleted` | Subscription canceled |
| `invoice.paid` | Invoice payment succeeded |
| `invoice.payment_failed` | Invoice payment failed |

### Customer Events

| Event | When it fires |
|-------|---------------|
| `customer.created` | New customer created |
| `payment_method.attached` | Payment method added to customer |

## Webhook Event Structure

Every webhook event has the same structure:

```json
{
  "id": "evt_1234567890",
  "object": "event",
  "type": "payment_intent.succeeded",
  "created": 1234567890,
  "data": {
    "object": {
      // The actual Stripe object (PaymentIntent, Invoice, etc.)
      "id": "pi_1234567890",
      "amount": 2000,
      "currency": "usd",
      // ... more fields
    }
  }
}
```

### Key Fields

- **`id`**: Unique event identifier (useful for idempotency)
- **`type`**: The event type (e.g., `payment_intent.succeeded`)
- **`data.object`**: The actual Stripe object that triggered the event

## Signature Verification - Critical for Security!

Anyone could send a POST request to your webhook endpoint. **You must verify the request came from Stripe.**

### How Signature Verification Works

1. Stripe signs each webhook with a secret key
2. The signature is included in the `Stripe-Signature` header
3. Your server verifies the signature before processing

```python
import stripe

# Your webhook secret (from Dashboard or stripe listen)
endpoint_secret = 'whsec_...'

# Verify the signature
event = stripe.Webhook.construct_event(
    payload,      # Raw request body
    sig_header,   # Stripe-Signature header
    endpoint_secret
)
```

### What Happens Without Verification?

An attacker could:
- Send fake "payment_intent.succeeded" events
- Grant themselves access to paid features
- Trigger unintended actions in your system

## Best Practices

### 1. Always Verify Signatures

Never skip signature verification in production!

### 2. Respond Quickly (< 30 seconds)

Stripe expects a response within 30 seconds. For long-running tasks:
- Return `200 OK` immediately
- Process asynchronously (queue, background job)

### 3. Handle Idempotency

Stripe may send the same event multiple times. Use `event.id` to detect duplicates:

```python
if already_processed(event['id']):
    return 200  # Already handled
```

### 4. Use Test Mode for Development

The Stripe CLI (`stripe listen`) creates a test webhook secret for local development.

### 5. Log Everything

Log all received events for debugging and audit purposes.

## Summary

In this introduction, you learned:

- **Webhooks** push notifications from Stripe to your server
- **Common events** include `payment_intent.succeeded` and subscription events
- **Signature verification** is critical for security
- **Best practices** include quick responses and idempotency handling

## Next Steps

Continue to `02_build_server.ipynb` to build a webhook server!